# NetShift-IDS · Colab launcher

This notebook orchestrates the repository pipeline, checks prerequisites, and copies environment and result artifacts to Google Drive. It contains no pipeline implementation.

Run the cells from top to bottom. Before starting, add `GITHUB_TOKEN` and `KAGGLE_API_TOKEN` to Colab Secrets, then place CIC-IDS2017 CSVs in `MyDrive/netshift-data/cicids2017/`. CSE-CIC-IDS2018 is downloaded from Kaggle dataset `solarmainframe/ids-intrusion-csv` and staged automatically. The private repository is cloned into the temporary Colab runtime; saved artifacts go to Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from google.colab import userdata

REPO_URL = 'https://github.com/krishnarawatsf/netshift-ids.git'
REPO_DIR = Path('/content/netshift-ids')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATASET_DRIVE_ROOT = DRIVE_ROOT / 'netshift-data'
RESULTS_DRIVE_ROOT = DRIVE_ROOT / 'netshift-results'
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')

def run_checked(command, *, cwd=None, env=None, stdout=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    subprocess.run(command, cwd=cwd, env=env, stdout=stdout, check=True)

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is missing. Add a repository-read token in Colab Secrets and rerun this cell.')

askpass_path = Path('/tmp/netshift_git_askpass.py')
askpass_path.write_text(
    'import os, sys\n'
    'prompt = sys.argv[1].lower()\n'
    "print('x-access-token' if 'username' in prompt else os.environ['GITHUB_TOKEN'])\n",
    encoding='utf-8',
)
git_env = os.environ.copy()
git_env['GITHUB_TOKEN'] = token
git_env['GIT_ASKPASS'] = str(askpass_path)
git_env['GIT_TERMINAL_PROMPT'] = '0'
try:
    if REPO_DIR.exists():
        if not (REPO_DIR / '.git').is_dir():
            raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository; remove or rename it and rerun.')
        run_checked(['git', 'pull', '--ff-only'], cwd=REPO_DIR, env=git_env)
    else:
        run_checked(['git', 'clone', REPO_URL, str(REPO_DIR)], env=git_env)
finally:
    askpass_path.unlink(missing_ok=True)

run_checked(['git', 'log', '--oneline', '-1'], cwd=REPO_DIR)
COMMIT = subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR,
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f'Run identity: {RUN_TIMESTAMP}-{COMMIT}')

In [ ]:
import subprocess

RESULTS_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
freeze_path = RESULTS_DRIVE_ROOT / f'pip-freeze-{RUN_TIMESTAMP}.txt'

run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=REPO_DIR)
run_checked([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR)
run_checked([sys.executable, '-m', 'pip', 'install', 'kagglehub==1.0.3'], cwd=REPO_DIR)
with freeze_path.open('w', encoding='utf-8') as freeze_file:
    run_checked([sys.executable, '-m', 'pip', 'freeze'], stdout=freeze_file)
print(f'Environment snapshot saved to {freeze_path}')

In [ ]:
import os
from pathlib import Path

def link_dataset(name):
    source = DATASET_DRIVE_ROOT / name
    target = REPO_DIR / 'data' / 'raw' / name
    if not source.is_dir():
        raise FileNotFoundError(
            f'Drive dataset folder is missing: {source}. '
            f'Create it and add the {name} CSV files before continuing.'
        )
    if target.is_symlink():
        if target.resolve() == source.resolve():
            print(f'{name}: Drive folder already linked.')
            return
        target.unlink()
    elif target.exists():
        if not target.is_dir():
            raise RuntimeError(f'Cannot link {name}: destination exists and is not a directory: {target}')
        entries = list(target.iterdir())
        placeholders = [entry for entry in entries if entry.name == '.gitkeep' and entry.is_file()]
        if len(entries) == len(placeholders):
            for placeholder in placeholders:
                placeholder.unlink()
            target.rmdir()
        elif entries:
            raise RuntimeError(f'Cannot link {name}: destination is not empty: {target}')
        else:
            target.rmdir()
    target.symlink_to(source, target_is_directory=True)
    print(f'{name}: linked {source} -> {target}')

link_dataset('cicids2017')

from google.colab import userdata
import kagglehub

KAGGLE_DATASET = 'solarmainframe/ids-intrusion-csv'
kaggle_token = userdata.get('KAGGLE_API_TOKEN')
if not kaggle_token:
    raise RuntimeError('Colab Secret KAGGLE_API_TOKEN is missing. Add your Kaggle API token in Colab Secrets and rerun this cell.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token
downloaded_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
csv_files = sorted(downloaded_dir.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Kaggle dataset {KAGGLE_DATASET} downloaded to {downloaded_dir}, but contains no CSV files.')

cse_target = REPO_DIR / 'data' / 'raw' / 'cse2018'
if cse_target.exists():
    existing = list(cse_target.iterdir())
    placeholders = [entry for entry in existing if entry.name == '.gitkeep' and entry.is_file()]
    if len(existing) == len(placeholders):
        for placeholder in placeholders:
            placeholder.unlink()
        if cse_target.exists():
            cse_target.rmdir()
    elif existing:
        expected = {str(path.resolve()) for path in csv_files}
        staged = {str(path.resolve()) for path in existing if path.is_symlink()}
        if len(existing) == len(csv_files) and staged == expected:
            print(f'CSE-CIC-IDS2018 already staged in {cse_target}')
        else:
            raise RuntimeError(f'Cannot stage Kaggle CSVs: destination is not empty: {cse_target}')
    else:
        cse_target.rmdir()
if not cse_target.exists():
    cse_target.mkdir(parents=True)
    names = [path.name for path in csv_files]
    if len(names) != len(set(names)):
        raise RuntimeError('Kaggle dataset contains duplicate CSV filenames; cannot stage them safely.')
    for source in csv_files:
        (cse_target / source.name).symlink_to(source)
    print(f'Staged {len(csv_files)} CSE-CIC-IDS2018 CSV files in {cse_target}')

In [ ]:
raw_dir = REPO_DIR / 'data' / 'raw'
synthetic_files = list(raw_dir.rglob('synthetic_part1.csv'))
if synthetic_files:
    raise RuntimeError(
        'Synthetic input detected in data/raw; refusing to run the experiment: '
        + ', '.join(str(path) for path in synthetic_files)
    )
print('Synthetic-file check passed.')

In [ ]:
doctor_script = REPO_DIR / 'scripts' / 'doctor.py'
if not doctor_script.is_file():
    raise FileNotFoundError(f'Required environment check is missing: {doctor_script}')
run_checked([sys.executable, doctor_script], cwd=REPO_DIR)
run_checked([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR)
print('Environment check and tests passed.')

In [ ]:
import json

def run_script(name, *args):
    script_path = REPO_DIR / 'scripts' / name
    if not script_path.is_file():
        raise FileNotFoundError(f'Required pipeline script is missing: {script_path}')
    run_checked([sys.executable, script_path, *args], cwd=REPO_DIR)

run_script('01_clean.py', 'cicids2017')
run_script('01_clean.py', 'cse2018', '--sample')
for dataset in ('cicids2017', 'cse2018'):
    report_path = REPO_DIR / 'results' / f'data_quality_{dataset}.json'
    if not report_path.is_file():
        raise FileNotFoundError(f'Expected cleaning report was not created: {report_path}')
    print(f'--- {report_path.name} ---')
    print(json.dumps(json.loads(report_path.read_text(encoding='utf-8')), indent=2))

In [ ]:
run_script('02_baseline.py')
run_script('03_cross_network.py')

In [ ]:
run_script('04_shap.py')

In [ ]:
run_script('05_label_budget.py')
run_script('05_label_budget.py', '--feature-set', 'shortcut_removed')

In [ ]:
run_script('07_active_labels.py')
run_script('06_report.py')
tables_path = REPO_DIR / 'results' / 'tables.md'
if not tables_path.is_file():
    raise FileNotFoundError(f'Report script did not create the expected table: {tables_path}')
print(tables_path.read_text(encoding='utf-8'))

In [ ]:
import shutil

source_results = REPO_DIR / 'results'
if not source_results.is_dir():
    raise FileNotFoundError(f'Results directory is missing: {source_results}')
run_destination = RESULTS_DRIVE_ROOT / f'{RUN_TIMESTAMP}-{COMMIT}'
if run_destination.exists():
    raise FileExistsError(f'Results destination already exists: {run_destination}')
shutil.copytree(source_results, run_destination / 'results')
print(f'Run results saved to {run_destination / "results"}')